# Stage 5: Model Bank 最終提交

讀取 Stage 3/4 artifacts，進行 minority augmentation、T2 mini model bank search，產生 final submission 候選。

預設 Drive root：`/content/drive/MyDrive/VeriPromiseESG2026`。


## 0. Colab 執行環境與路徑檢查

請先執行下一個 setup cell。它會安裝套件、掛載 Google Drive、檢查 GPU、建立輸出資料夾，並驗證必要輸入檔是否存在。


In [ ]:
# Colab 啟動區塊：安裝套件、掛載 Drive、檢查 GPU、驗證必要輸入檔。
!pip -q install transformers accelerate scikit-learn pandas numpy tqdm openpyxl

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
STAGE_NAME = "stage5_model_bank_final_submission"
STAGE_DISPLAY_NAME = "Stage 5 - Model Bank 最終提交"
OUTPUT_DIR = f"{OUTPUT_ROOT}/{STAGE_NAME}"
OUT_DIR = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUIRED_RELATIVE_PATHS = [
    "data/vpesg4k_train_1000.json",
    "data/vpesg4k_valdata_1000.json",
    "data/vpesg4k_testdata_2000.json",
    "outputs/stage3_tta_t4_specialist_blend/stage3_t4_specialist_probs.pkl",
    "outputs/stage3_tta_t4_specialist_blend/stage3_3view_probs.pkl",
    "outputs/stage3_tta_t4_specialist_blend/stage3_best_config.json",
    "outputs/stage4_t2_specialist_calibration/stage4_t2_specialist_probs.pkl",
    "outputs/stage4_t2_specialist_calibration/stage4_t2_best_config.json",
    "outputs/stage4_t2_specialist_calibration/stage4_conservative_config.json",
    "outputs/stage4_t2_specialist_calibration/stage4_config.json",
]

def stage_log(label, value):
    print(f"[{STAGE_NAME}] {label}: {value}")

def require_files(relative_paths):
    missing = []
    for rel in relative_paths:
        path = f"{BASE_DIR}/{rel}"
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        msg = "缺少必要輸入檔。請先執行前一個 Stage，或將資料放到 Google Drive：\n" + "\n".join(missing)
        raise FileNotFoundError(msg)

require_files(REQUIRED_RELATIVE_PATHS)
stage_log("Stage", STAGE_DISPLAY_NAME)
stage_log("BASE_DIR", BASE_DIR)
stage_log("DATA_DIR", DATA_DIR)
stage_log("OUTPUT_DIR", OUTPUT_DIR)

# 完整訓練建議使用 Colab A100 或同級 GPU。
try:
    gpu_names = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_names = list(gpu_names)
    stage_log("GPU", gpu_names)
    if not any("A100" in str(name) for name in gpu_names):
        stage_log("WARNING", "建議使用 A100；非 A100 runtime 可能需要更長時間。")
except Exception as exc:
    stage_log("WARNING", f"無法取得 GPU 資訊，請確認 Colab runtime 已啟用 GPU。{exc}")


## 輸入輸出契約

Stage 顯示名稱：`Stage 5 - Model Bank 最終提交`

Google Drive 輸出資料夾：

```text
/content/drive/MyDrive/VeriPromiseESG2026/outputs/stage5_model_bank_final_submission/
```

必要輸入檔：

- `data/vpesg4k_train_1000.json`
- `data/vpesg4k_valdata_1000.json`
- `data/vpesg4k_testdata_2000.json`
- `outputs/stage3_tta_t4_specialist_blend/stage3_t4_specialist_probs.pkl`
- `outputs/stage3_tta_t4_specialist_blend/stage3_3view_probs.pkl`
- `outputs/stage3_tta_t4_specialist_blend/stage3_best_config.json`
- `outputs/stage4_t2_specialist_calibration/stage4_t2_specialist_probs.pkl`
- `outputs/stage4_t2_specialist_calibration/stage4_t2_best_config.json`
- `outputs/stage4_t2_specialist_calibration/stage4_conservative_config.json`
- `outputs/stage4_t2_specialist_calibration/stage4_config.json`

主要輸出檔：

- `stage5_t2_aug_specialist_probs.pkl`
- `stage5_augmentation_config.json`
- `stage5_config.json`
- `stage5_best_val_test2000_submission.csv`
- `stage5_low_risk_high_score_test2000_submission.csv`
- `stage5_safe_test2000_submission.csv`

若必要輸入不存在，第一個 setup cell 會直接列出缺少的路徑並停止。


## Stage 主要流程

本節整合 Stage 3/4 的 probability sources，動態過濾不可用候選，產生最終 submission 候選。


## 1. 路徑與全域設定

In [ ]:

import os
import json
import pickle
import random
import re
import math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V32_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_T4_PROB_PATH = f"{V32_DIR}/stage3_t4_specialist_probs.pkl"

V33_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V33_VIEW_PROB_PATH = f"{V33_DIR}/stage3_3view_probs.pkl"

V34_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V34_CONFIG_PATH = f"{V34_DIR}/stage3_best_config.json"

V35_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_T2_PROB_PATH = f"{V35_DIR}/stage4_t2_specialist_probs.pkl"
V35_CONFIG_PATH = f"{V35_DIR}/stage4_t2_best_config.json"

V35_01_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_01_CONFIG_PATH = f"{V35_01_DIR}/stage4_conservative_config.json"

V36_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V36_CONFIG_PATH = f"{V36_DIR}/stage4_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage5_model_bank_final_submission"
os.makedirs(OUT_DIR, exist_ok=True)

T2_AUG_PROB_PATH = f"{OUT_DIR}/stage5_t2_aug_specialist_probs.pkl"
T4_AUG_PROB_PATH = f"{OUT_DIR}/stage5_t4_aug_specialist_probs.pkl"

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T2_LABELS = ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years"]
T4_LABELS = ["Clear", "Not Clear", "Misleading"]

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

T2_LABEL2ID = {lab: i for i, lab in enumerate(T2_LABELS)}
T2_ID2LABEL = {i: lab for lab, i in T2_LABEL2ID.items()}

T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

MODEL_NAME = "hfl/chinese-macbert-base"

N_SPLITS = 5
MAX_LEN = 384
EPOCHS = 3
BATCH_SIZE = 8
PRED_BATCH_SIZE = 16
GRAD_ACCUM = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 1
SEED = 46

TRAIN_VIEWS = ["head", "tail"]
PRED_VIEWS = ["head", "middle", "tail"]

# synthetic 樣本不要壓過真實資料
SYN_SAMPLE_WEIGHT_T2 = 0.35
SYN_SAMPLE_WEIGHT_T4 = 0.25

USE_CLASS_WEIGHTS = True
CLASS_WEIGHT_MAX_T2 = 8.0
CLASS_WEIGHT_MAX_T4 = 10.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)
for p in [
    TRAIN_PATH, VAL_PATH, TEST_PATH,
    V33_VIEW_PROB_PATH, V32_T4_PROB_PATH, V34_CONFIG_PATH,
    V35_T2_PROB_PATH, V35_CONFIG_PATH, V35_01_CONFIG_PATH, V36_CONFIG_PATH
]:
    print(os.path.exists(p), p)


In [ ]:

def seed_everything(seed=46):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


## 2. 讀取資料與既有快取

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(V34_CONFIG_PATH, "r", encoding="utf-8") as f:
    V34_CONFIG_OBJ = json.load(f)
V34_BASE_CONFIG = V34_CONFIG_OBJ["base_config"]
V34_T4_ALPHA = float(V34_CONFIG_OBJ["t4_alpha"])
V34_T4_SPEC_MULT = V34_CONFIG_OBJ["t4_spec_multipliers"]

with open(V33_VIEW_PROB_PATH, "rb") as f:
    view_obj = pickle.load(f)

with open(V32_T4_PROB_PATH, "rb") as f:
    t4_obj = pickle.load(f)

t4_val_probs_old = t4_obj["val_probs"]
t4_test_probs_old = t4_obj["test_probs"]

with open(V35_T2_PROB_PATH, "rb") as f:
    t2_obj_old = pickle.load(f)

with open(V35_CONFIG_PATH, "r", encoding="utf-8") as f:
    V35_CONFIG_OBJ = json.load(f)

with open(V35_01_CONFIG_PATH, "r", encoding="utf-8") as f:
    V35_01_CONFIG_OBJ = json.load(f)

with open(V36_CONFIG_PATH, "r", encoding="utf-8") as f:
    V36_CONFIG_OBJ = json.load(f)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nTrain label distributions:")
for t in TASKS:
    print(t, train_df[t].value_counts().to_dict())

print("\nVal label distributions:")
for t in TASKS:
    print(t, val_df[t].value_counts().to_dict())


## 3. 產生 minority synthetic seeds

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

existing_texts = set()
for df in [train_df, val_df, test_df]:
    for x in df["data"].astype(str).tolist():
        existing_texts.add(compact_text(x))

companies = [
    "本公司", "集團", "公司", "廠區", "營運據點", "供應鏈管理單位",
    "永續發展委員會", "環安衛部門", "人資單位", "資訊安全團隊"
]

topics = [
    "溫室氣體盤查", "能源管理", "供應商稽核", "職業安全衛生", "資安管理",
    "水資源管理", "廢棄物減量", "員工教育訓練", "人權盡職調查", "客戶隱私保護",
    "產品碳足跡", "再生能源使用", "社區參與計畫", "內部控制制度"
]

years_within2 = ["2025", "2026"]
years_between25 = ["2027", "2028", "2029"]
years_more5 = ["2030", "2035", "2040", "2050"]

def make_row(text, promise_status, verification_timeline, evidence_status, evidence_quality, target_reason, source_type, sample_weight):
    return {
        "id": f"synthetic_{source_type}_{abs(hash(text)) % 10**10}",
        "data": text,
        "company": "synthetic",
        "page_number": -1,
        "promise_status": promise_status,
        "verification_timeline": verification_timeline,
        "evidence_status": evidence_status,
        "evidence_quality": evidence_quality,
        "target_reason": target_reason,
        "source_type": source_type,
        "sample_weight": sample_weight,
    }

def generate_t2_within2(n=80):
    rows = []
    templates = [
        "{company}已訂定{topic}短期改善計畫，預計於{year}年完成導入並檢視執行成果。",
        "{company}將於{year}年前完成{topic}相關制度建置，並於完成後定期追蹤改善情形。",
        "針對{topic}，{company}規劃於兩年內完成內部作業流程優化與成效追蹤。",
        "{company}預計在{year}年完成{topic}之教育訓練與管理機制更新。",
        "為強化{topic}，{company}將於二年內完成盤點、改善與後續追蹤作業。",
        "{company}已啟動{topic}專案，預計明年完成第一階段改善並揭露成果。",
        "{company}規劃於後年完成{topic}查核機制建立，作為短期永續管理目標。",
    ]
    for i in range(n):
        text = templates[i % len(templates)].format(
            company=companies[i % len(companies)],
            topic=topics[(i * 3) % len(topics)],
            year=years_within2[i % len(years_within2)],
        )
        rows.append(make_row(
            text, "Yes", "within_2_years", "No", "N/A",
            "含兩年內/2025/2026/明年/後年等短期完成線索",
            "syn_t2_within2",
            SYN_SAMPLE_WEIGHT_T2,
        ))
    return rows

def generate_t2_more5(n=80):
    rows = []
    templates = [
        "{company}承諾於{year}年前達成{topic}長期目標，並逐年追蹤執行情形。",
        "為回應淨零趨勢，{company}設定{year}年完成{topic}轉型目標。",
        "{company}已規劃長期{topic}藍圖，目標於{year}年前完成主要里程碑。",
        "{company}將持續推動{topic}，並以{year}年作為長程改善目標年。",
        "針對{topic}，{company}訂定至{year}年的中長期行動方案。",
        "{company}以{year}年為目標，推動{topic}相關減量與管理措施。",
        "{company}承諾配合RE100與淨零路徑，於{year}年前提升{topic}績效。",
    ]
    for i in range(n):
        text = templates[i % len(templates)].format(
            company=companies[i % len(companies)],
            topic=topics[(i * 5) % len(topics)],
            year=years_more5[i % len(years_more5)],
        )
        rows.append(make_row(
            text, "Yes", "more_than_5_years", "No", "N/A",
            "含2030/2035/2040/2050/長期/淨零等長期目標線索",
            "syn_t2_more5",
            SYN_SAMPLE_WEIGHT_T2,
        ))
    return rows

def generate_t4_not_clear(n=120):
    rows = []
    templates = [
        "{company}持續推動{topic}，並透過內部會議追蹤相關成果，未來將視執行情形滾動調整。",
        "{company}積極落實{topic}管理，持續強化員工意識並提升整體執行效率。",
        "為提升{topic}表現，{company}定期檢視制度並持續優化相關流程。",
        "{company}致力於改善{topic}，透過跨部門合作逐步推動各項改善方案。",
        "{company}已建立{topic}管理方向，並將持續推廣相關措施以提升永續績效。",
        "{company}透過教育宣導與日常管理推動{topic}，但尚未揭露具體量化成果。",
        "針對{topic}，{company}規劃後續改善措施，並持續追蹤各單位執行情形。",
    ]
    for i in range(n):
        text = templates[i % len(templates)].format(
            company=companies[i % len(companies)],
            topic=topics[(i * 2) % len(topics)],
        )
        rows.append(make_row(
            text, "Yes", "already", "Yes", "Not Clear",
            "有執行描述但缺乏明確數據、外部查證或具體成效",
            "syn_t4_not_clear",
            SYN_SAMPLE_WEIGHT_T4,
        ))
    return rows

def generate_t4_misleading(n=80):
    rows = []
    templates = [
        "{company}表示已完成{topic}改善，但段落僅描述政策宣導，未說明改善對象、衡量方法或實際成果。",
        "{company}宣稱{topic}已達成目標，惟文字僅提到內部規範，未揭露可驗證的數據或證明文件。",
        "針對{topic}，{company}提及取得相關成效，但所列內容與承諾目標不直接對應，無法確認是否真正完成。",
        "{company}說明{topic}已受查核，但查核範圍並未涵蓋該承諾項目，因此證據與承諾不完全匹配。",
        "{company}以一般活動成果作為{topic}達成證明，但未說明該活動如何支持原先承諾。",
        "{company}揭露{topic}相關數字，但數字僅為參與人次，無法證明承諾中的改善或減量成果。",
        "{company}將願景目標描述為已完成成果，惟未提供實際執行證據，容易造成讀者誤解。",
    ]
    for i in range(n):
        text = templates[i % len(templates)].format(
            company=companies[i % len(companies)],
            topic=topics[(i * 4) % len(topics)],
        )
        rows.append(make_row(
            text, "Yes", "already", "Yes", "Misleading",
            "文字看似有證據，但證據與承諾不匹配或不足以支持結論",
            "syn_t4_misleading",
            SYN_SAMPLE_WEIGHT_T4,
        ))
    return rows

synthetic_rows = []
synthetic_rows += generate_t2_within2(80)
synthetic_rows += generate_t2_more5(80)
synthetic_rows += generate_t4_not_clear(120)
synthetic_rows += generate_t4_misleading(80)

syn_df = pd.DataFrame(synthetic_rows)
print("raw synthetic:", syn_df.shape)
display(syn_df.head())
display(syn_df["source_type"].value_counts())


## 4. Synthetic Quality Gate

In [ ]:

def hierarchy_valid(row):
    if row["promise_status"] == "No":
        return row["verification_timeline"] == "N/A" and row["evidence_status"] == "N/A" and row["evidence_quality"] == "N/A"
    if row["evidence_status"] != "Yes":
        return row["evidence_quality"] == "N/A"
    return True

def has_t2_cue(row):
    text = str(row["data"])
    vt = row["verification_timeline"]
    if vt == "within_2_years":
        return bool(re.search(r"兩年內|二年內|2年內|2025|2026|明年|後年|短期|近期", text))
    if vt == "more_than_5_years":
        return bool(re.search(r"2030|2035|2040|2050|長期|長程|淨零|碳中和|RE100", text))
    return True

def has_t4_cue(row):
    text = str(row["data"])
    eq = row["evidence_quality"]
    if eq == "Not Clear":
        return bool(re.search(r"持續|積極|逐步|強化|提升|致力|定期檢視|持續優化|尚未揭露|規劃", text))
    if eq == "Misleading":
        return bool(re.search(r"宣稱|未說明|未揭露|不直接對應|不完全匹配|無法確認|誤解|無法證明", text))
    return True

def quality_gate(row):
    text = str(row["data"])
    if len(text) < 35 or len(text) > 260:
        return False
    if compact_text(text) in existing_texts:
        return False
    if not hierarchy_valid(row):
        return False
    if not has_t2_cue(row):
        return False
    if not has_t4_cue(row):
        return False
    return True

syn_df["pass_quality_gate"] = syn_df.apply(quality_gate, axis=1)
syn_pass_df = syn_df[syn_df["pass_quality_gate"]].drop_duplicates(subset=["data"]).reset_index(drop=True)

print("pass synthetic:", syn_pass_df.shape)
display(syn_pass_df["source_type"].value_counts())
display(syn_pass_df.groupby(["verification_timeline", "evidence_quality"]).size())

syn_df.to_csv(f"{OUT_DIR}/stage5_synthetic_aug_all.csv", index=False, encoding="utf-8-sig")
syn_pass_df.to_csv(f"{OUT_DIR}/stage5_synthetic_aug_pass.csv", index=False, encoding="utf-8-sig")

print("saved:")
print(f"{OUT_DIR}/stage5_synthetic_aug_all.csv")
print(f"{OUT_DIR}/stage5_synthetic_aug_pass.csv")


## 5. 評分與重建既有 pipeline

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

T2_VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

# Compatibility aliases for older Stage3/Stage4 view preset names.
def resolve_view_preset(presets, preset_name, *, default="head", context="view preset"):
    preset_name = default if preset_name is None else preset_name
    aliases = {
        "head_middle": "head_mid",
        "middle_tail": "mid_tail",
    }
    preset_name = aliases.get(preset_name, preset_name)
    if preset_name not in presets:
        available = ", ".join(sorted(presets.keys()))
        raise KeyError(f"Unknown {context}: {preset_name}. Available presets: {available}")
    return presets[preset_name]

def mix_views(stem_view_probs, split, task, preset_name):
    preset = resolve_view_preset(VIEW_PRESETS, preset_name, context="Stage3/Stage4 view preset")
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_config_as_pred(df, split, config):
    probs = build_tta_probs(split, config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(df, probs, post_cfg)
    return pred, probs

def t4_adjust_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_quality_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]

    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]

    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def get_t4_blend_probs(base_probs, base_config, spec_probs, alpha=0.05, spec_multipliers=None):
    base_q3 = get_base_quality_non_na_probs(base_probs, base_config)
    spec_q3 = t4_adjust_probs(spec_probs, spec_multipliers=spec_multipliers)
    blend = (1 - alpha) * base_q3 + alpha * spec_q3
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
    return blend.astype(np.float32)

def compose_quality_from_p3(base_pred, q3):
    out = base_pred.copy()
    mask = out["evidence_status"] == "Yes"
    ids = q3.argmax(axis=1)
    labels = np.array([T4_ID2LABEL[int(i)] for i in ids])
    out.loc[mask, "evidence_quality"] = labels[mask.values]
    out.loc[~mask, "evidence_quality"] = "N/A"
    return apply_hierarchy_to_pred_df(out)

def build_v34_base(df, split):
    base_pred, base_probs = eval_config_as_pred(df, split, V34_BASE_CONFIG)
    t4_probs = t4_val_probs_old if split == "val" else t4_test_probs_old
    q3 = get_t4_blend_probs(
        base_probs,
        V34_BASE_CONFIG,
        t4_probs,
        alpha=V34_T4_ALPHA,
        spec_multipliers=V34_T4_SPEC_MULT,
    )
    pred = compose_quality_from_p3(base_pred, q3)
    return pred, base_probs, q3

def mix_t2_views(t2_obj, split, preset_name):
    preset = resolve_view_preset(T2_VIEW_PRESETS, preset_name, context="T2 view preset")
    prob_dict = t2_obj[f"{split}_probs"]

    out = None
    for view, w in preset.items():
        p = prob_dict[view]
        out = w * p if out is None else out + w * p

    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def adjust_t2_spec_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T2_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_timeline_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    p = adjusted["verification_timeline"]

    non_na_ids = [
        LABEL2ID["verification_timeline"]["already"],
        LABEL2ID["verification_timeline"]["within_2_years"],
        LABEL2ID["verification_timeline"]["between_2_and_5_years"],
        LABEL2ID["verification_timeline"]["more_than_5_years"],
    ]

    p4 = p[:, non_na_ids].copy()
    p4 = p4 / np.clip(p4.sum(axis=1, keepdims=True), 1e-12, None)
    return p4

def get_t2_blend_probs(t2_obj, split, base_probs, alpha, view_preset, spec_multipliers):
    base_p4 = get_base_timeline_non_na_probs(base_probs, V34_BASE_CONFIG)
    spec_p4 = mix_t2_views(t2_obj, split, view_preset)
    spec_p4 = adjust_t2_spec_probs(spec_p4, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_p4 + alpha * spec_p4
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
    return blend.astype(np.float32)

def compose_timeline_from_p4(base_pred, p4):
    out = base_pred.copy()
    mask = out["promise_status"] == "Yes"
    ids = p4.argmax(axis=1)
    labels = np.array([T2_ID2LABEL[int(i)] for i in ids])
    out.loc[mask, "verification_timeline"] = labels[mask.values]
    out.loc[~mask, "verification_timeline"] = "N/A"
    return apply_hierarchy_to_pred_df(out)

def normalize_stage4_base_choice(base_choice):
    aliases = {
        "stage3": "v34",
        "v34": "v34",
        "stage4_distribution_matched": "stage4_conservative_distribution_matched",
        "stage4_safe": "stage4_conservative_safe",
        "stage4_very_safe": "stage4_conservative_very_safe",
        "stage4_conservative_distribution_matched": "stage4_conservative_distribution_matched",
        "stage4_conservative_best_val": "stage4_conservative_best_val",
        "stage4_conservative_safe": "stage4_conservative_safe",
        "stage4_conservative_very_safe": "stage4_conservative_very_safe",
        "stage4_t2_best_val": "stage4_t2_best_val",
    }
    return aliases.get(base_choice, base_choice)

def is_stage4_base_choice_available(base_choice):
    normalized = normalize_stage4_base_choice(base_choice)
    if normalized in {"v34", "stage4_t2_best_val"}:
        return True
    if normalized.startswith("stage4_conservative_"):
        name = normalized.replace("stage4_conservative_", "")
        return V35_01_CONFIG_OBJ.get("choices", {}).get(name) is not None
    return False

def filter_available_stage4_base_choices(candidates):
    available, skipped = [], []
    for choice in candidates:
        if is_stage4_base_choice_available(choice):
            available.append(choice)
        else:
            skipped.append(choice)
    if skipped:
        print("Skipping unavailable Stage 4 base choices:", skipped)
    if not available:
        raise RuntimeError("No available Stage 4 base choices. Check Stage 4 config outputs first.")
    return available

def get_t2_cfg_from_choice(base_choice):
    original_choice = base_choice
    base_choice = normalize_stage4_base_choice(base_choice)
    if base_choice == "stage4_t2_best_val":
        return {
            "view": V35_CONFIG_OBJ["best_view"],
            "alpha": V35_CONFIG_OBJ["best_alpha"],
            "spec_multipliers": V35_CONFIG_OBJ["best_spec_multipliers"],
        }
    if base_choice.startswith("stage4_conservative_"):
        name = base_choice.replace("stage4_conservative_", "")
        cfg = V35_01_CONFIG_OBJ.get("choices", {}).get(name)
        if cfg is None:
            raise ValueError(f"choice not found: {base_choice}")
        return {
            "view": cfg["view"],
            "alpha": cfg["alpha"],
            "spec_multipliers": cfg["spec_multipliers"],
        }
    if base_choice == "v34":
        return {
            "view": "head",
            "alpha": 0.0,
            "spec_multipliers": None,
        }
    raise ValueError(base_choice)

def build_existing_pipeline_pred(df, split, base_choice):
    v34_pred, base_probs, q3 = build_v34_base(df, split)
    t2_cfg = get_t2_cfg_from_choice(base_choice)
    p4 = get_t2_blend_probs(
        t2_obj_old,
        split=split,
        base_probs=base_probs,
        alpha=t2_cfg["alpha"],
        view_preset=t2_cfg["view"],
        spec_multipliers=t2_cfg["spec_multipliers"],
    )
    pred = compose_timeline_from_p4(v34_pred, p4)
    pred = compose_quality_from_p3(pred, q3)
    return pred, base_probs, p4, q3


## 6. Specialist model / dataset

In [ ]:

class SpecialistModel(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden * 2, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        last_hidden = out.last_hidden_state

        cls_vec = last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        pooled = torch.cat([cls_vec, mean_vec], dim=-1)
        logits = self.classifier(self.dropout(pooled))
        return logits

class SpecialistDataset(Dataset):
    def __init__(self, df, tokenizer, label_col=None, label2id=None, views=None, max_len=384):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.label_col = label_col
        self.label2id = label2id

        self.cls_id = getattr(tokenizer, "cls_token_id", None)
        self.sep_id = getattr(tokenizer, "sep_token_id", None)
        self.pad_id = getattr(tokenizer, "pad_token_id", None)

        if self.cls_id is None:
            self.cls_id = tokenizer.convert_tokens_to_ids("[CLS]")
        if self.sep_id is None:
            self.sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
        if self.pad_id is None:
            self.pad_id = 0

        self.special_n = 2

        if views is None:
            views = ["head"]

        self.records = []
        df = df.reset_index(drop=True)

        for _, row in df.iterrows():
            text = str(row["data"])
            label = None
            if label_col is not None:
                label = int(label2id[row[label_col]])
            sample_weight = float(row.get("sample_weight", 1.0))
            for view in views:
                self.records.append((text, label, sample_weight, view))

    def __len__(self):
        return len(self.records)

    def select_tokens(self, token_ids, view):
        max_content_len = self.max_len - self.special_n
        if len(token_ids) <= max_content_len:
            return token_ids
        if view == "head":
            return token_ids[:max_content_len]
        if view == "tail":
            return token_ids[-max_content_len:]
        if view == "middle":
            start = max((len(token_ids) - max_content_len) // 2, 0)
            return token_ids[start:start + max_content_len]
        raise ValueError(view)

    def __getitem__(self, idx):
        text, label, sample_weight, view = self.records[idx]
        token_ids = self.tokenizer.encode(text, add_special_tokens=False)
        token_ids = self.select_tokens(token_ids, view)

        input_ids = [self.cls_id] + token_ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)
        token_type_ids = [0] * len(input_ids)

        pad_len = self.max_len - len(input_ids)
        if pad_len > 0:
            input_ids += [self.pad_id] * pad_len
            attention_mask += [0] * pad_len
            token_type_ids += [0] * pad_len
        else:
            input_ids = input_ids[:self.max_len]
            attention_mask = attention_mask[:self.max_len]
            token_type_ids = token_type_ids[:self.max_len]

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "sample_weight": torch.tensor(sample_weight, dtype=torch.float32),
        }

        if label is not None:
            item["labels"] = torch.tensor(label, dtype=torch.long)

        return item

def batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def make_class_weights(labels, id2label, max_weight):
    counts = Counter(labels)
    total = len(labels)
    weights = []
    for i in range(len(id2label)):
        c = counts.get(i, 1)
        w = total / (len(id2label) * c)
        weights.append(w)
    weights = np.array(weights, dtype=np.float32)
    weights = np.clip(weights, 1.0, max_weight)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)

@torch.no_grad()
def predict_probs_for_model(model, df, tokenizer, view, batch_size=16):
    model.eval()
    ds = SpecialistDataset(df, tokenizer, label_col=None, views=[view], max_len=MAX_LEN)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    all_probs = []
    for batch in tqdm(dl, desc=f"predict {view}", leave=False):
        batch = batch_to_device(batch)
        batch.pop("sample_weight", None)
        logits = model(**batch)
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        all_probs.append(probs)

    return np.concatenate(all_probs, axis=0).astype(np.float32)


## 7. 訓練通用 specialist

In [ ]:

def train_specialist_one_fold(
    fold,
    task_name,
    train_part,
    valid_part,
    tokenizer,
    label_col,
    labels,
    label2id,
    id2label,
    class_weight_max,
):
    print(f"\n===== {task_name} Fold {fold} =====")

    train_label_ids = train_part[label_col].map(label2id).values.astype(int)
    valid_label_ids = valid_part[label_col].map(label2id).values.astype(int)

    train_ds = SpecialistDataset(train_part, tokenizer, label_col=label_col, label2id=label2id, views=TRAIN_VIEWS, max_len=MAX_LEN)
    valid_ds = SpecialistDataset(valid_part, tokenizer, label_col=label_col, label2id=label2id, views=["head"], max_len=MAX_LEN)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    valid_dl = DataLoader(valid_ds, batch_size=PRED_BATCH_SIZE, shuffle=False, num_workers=0)

    model = SpecialistModel(MODEL_NAME, num_labels=len(labels), dropout=0.2).to(DEVICE)

    if USE_CLASS_WEIGHTS:
        class_weights = make_class_weights(train_label_ids, id2label, class_weight_max)
        print("class_weights:", {id2label[i]: float(class_weights[i].detach().cpu()) for i in range(len(labels))})
        criterion = nn.CrossEntropyLoss(weight=class_weights, reduction="none")
    else:
        criterion = nn.CrossEntropyLoss(reduction="none")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    total_steps = math.ceil(len(train_dl) / GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_score = -1
    best_path = f"{OUT_DIR}/{task_name}_fold{fold}_best.pt"
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_dl, desc=f"{task_name} fold {fold} epoch {epoch}", leave=False)

        for step, batch in enumerate(pbar, start=1):
            batch = batch_to_device(batch)
            labels_tensor = batch.pop("labels")
            sample_weight = batch.pop("sample_weight")

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(**batch)
                loss_vec = criterion(logits, labels_tensor)
                loss = (loss_vec * sample_weight).mean()
                loss = loss / GRAD_ACCUM

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM == 0 or step == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            losses.append(float(loss.detach().cpu()) * GRAD_ACCUM)
            pbar.set_postfix(loss=np.mean(losses[-20:]))

        valid_probs = predict_probs_for_model(model, valid_part, tokenizer, view="head", batch_size=PRED_BATCH_SIZE)
        valid_pred = valid_probs.argmax(axis=1)
        score = f1_score(valid_label_ids, valid_pred, labels=list(range(len(labels))), average="macro", zero_division=0)

        print(f"{task_name} fold {fold} epoch {epoch}: loss={np.mean(losses):.4f} val_macro_f1={score:.6f}")

        if score > best_score:
            best_score = score
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
            print("saved:", best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stop")
                break

    del model
    torch.cuda.empty_cache()

    return best_path, best_score

def train_and_predict_specialist(
    task_name,
    real_df,
    syn_task_df,
    label_col,
    labels,
    label2id,
    id2label,
    class_weight_max,
    prob_path,
):
    if os.path.exists(prob_path):
        print("Loading existing probs:", prob_path)
        with open(prob_path, "rb") as f:
            return pickle.load(f)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    real_df = real_df.reset_index(drop=True).copy()
    real_df["sample_weight"] = 1.0
    real_df["source_type"] = "real"

    syn_task_df = syn_task_df.reset_index(drop=True).copy()

    y = real_df[label_col].map(label2id).values.astype(int)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_paths = []
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(real_df, y), start=1):
        train_real = real_df.iloc[tr_idx].reset_index(drop=True)
        valid_real = real_df.iloc[va_idx].reset_index(drop=True)

        # synthetic 只進 train，不進 valid
        train_part = pd.concat([train_real, syn_task_df], axis=0, ignore_index=True)
        train_part = train_part.sample(frac=1.0, random_state=SEED + fold).reset_index(drop=True)

        path, score = train_specialist_one_fold(
            fold=fold,
            task_name=task_name,
            train_part=train_part,
            valid_part=valid_real,
            tokenizer=tokenizer,
            label_col=label_col,
            labels=labels,
            label2id=label2id,
            id2label=id2label,
            class_weight_max=class_weight_max,
        )
        fold_paths.append(path)
        fold_scores.append(score)

    print(task_name, "fold_scores:", fold_scores)
    print(task_name, "mean:", np.mean(fold_scores))

    view_probs = {"val": {}, "test": {}}

    for view in PRED_VIEWS:
        print(f"\n=== {task_name} predict view: {view} ===")
        val_fold_probs = []
        test_fold_probs = []

        for fold, path in enumerate(fold_paths, start=1):
            print("loading", path)
            model = SpecialistModel(MODEL_NAME, num_labels=len(labels), dropout=0.2).to(DEVICE)
            state = torch.load(path, map_location=DEVICE)
            model.load_state_dict(state)

            val_probs = predict_probs_for_model(model, val_df, tokenizer, view=view, batch_size=PRED_BATCH_SIZE)
            test_probs = predict_probs_for_model(model, test_df, tokenizer, view=view, batch_size=PRED_BATCH_SIZE)

            val_fold_probs.append(val_probs)
            test_fold_probs.append(test_probs)

            del model
            torch.cuda.empty_cache()

        view_probs["val"][view] = np.mean(val_fold_probs, axis=0).astype(np.float32)
        view_probs["test"][view] = np.mean(test_fold_probs, axis=0).astype(np.float32)

    obj = {
        "model_name": MODEL_NAME,
        "task_name": task_name,
        "fold_paths": fold_paths,
        "fold_scores": fold_scores,
        "train_views": TRAIN_VIEWS,
        "pred_views": PRED_VIEWS,
        "labels": labels,
        "val_probs": view_probs["val"],
        "test_probs": view_probs["test"],
        "val_ids": val_df["id"].tolist(),
        "test_ids": test_df["id"].tolist(),
        "synthetic_counts": syn_task_df["source_type"].value_counts().to_dict() if len(syn_task_df) else {},
    }

    with open(prob_path, "wb") as f:
        pickle.dump(obj, f)

    print("saved:", prob_path)
    return obj


## 8. 準備 T2 / T4 augmentation datasets

In [ ]:

# T2 specialist real data: promise_status=Yes 且 timeline 非 N/A
t2_real_df = train_df[
    (train_df["promise_status"] == "Yes") &
    (train_df["verification_timeline"].isin(T2_LABELS))
].reset_index(drop=True).copy()
t2_real_df["sample_weight"] = 1.0
t2_real_df["source_type"] = "real"

t2_syn_df = syn_pass_df[
    syn_pass_df["verification_timeline"].isin(["within_2_years", "more_than_5_years"])
].reset_index(drop=True).copy()

# T4 specialist real data: evidence_status=Yes 且 quality 非 N/A
t4_real_df = train_df[
    (train_df["evidence_status"] == "Yes") &
    (train_df["evidence_quality"].isin(T4_LABELS))
].reset_index(drop=True).copy()
t4_real_df["sample_weight"] = 1.0
t4_real_df["source_type"] = "real"

t4_syn_df = syn_pass_df[
    syn_pass_df["evidence_quality"].isin(["Not Clear", "Misleading"])
].reset_index(drop=True).copy()

print("T2 real:", t2_real_df.shape, t2_real_df["verification_timeline"].value_counts().to_dict())
print("T2 synthetic:", t2_syn_df.shape, t2_syn_df["verification_timeline"].value_counts().to_dict(), t2_syn_df["source_type"].value_counts().to_dict())

print("\nT4 real:", t4_real_df.shape, t4_real_df["evidence_quality"].value_counts().to_dict())
print("T4 synthetic:", t4_syn_df.shape, t4_syn_df["evidence_quality"].value_counts().to_dict(), t4_syn_df["source_type"].value_counts().to_dict())


## 9. 訓練 / 載入 augmented specialists

In [ ]:

t2_aug_obj = train_and_predict_specialist(
    task_name="t2_aug",
    real_df=t2_real_df,
    syn_task_df=t2_syn_df,
    label_col="verification_timeline",
    labels=T2_LABELS,
    label2id=T2_LABEL2ID,
    id2label=T2_ID2LABEL,
    class_weight_max=CLASS_WEIGHT_MAX_T2,
    prob_path=T2_AUG_PROB_PATH,
)

t4_aug_obj = train_and_predict_specialist(
    task_name="t4_aug",
    real_df=t4_real_df,
    syn_task_df=t4_syn_df,
    label_col="evidence_quality",
    labels=T4_LABELS,
    label2id=T4_LABEL2ID,
    id2label=T4_ID2LABEL,
    class_weight_max=CLASS_WEIGHT_MAX_T4,
    prob_path=T4_AUG_PROB_PATH,
)

print("T2 aug fold scores:", t2_aug_obj.get("fold_scores"))
print("T4 aug fold scores:", t4_aug_obj.get("fold_scores"))


## 10. Augmented specialists 搜尋與合併

In [ ]:

def mix_specialist_views(obj, split, preset_name, label_kind):
    preset = resolve_view_preset(T2_VIEW_PRESETS, preset_name, context="T2 view preset")
    prob_dict = obj[f"{split}_probs"]
    out = None
    for view, w in preset.items():
        p = prob_dict[view]
        out = w * p if out is None else out + w * p
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def adjust_p4(p4, mult):
    p = p4.copy()
    if mult:
        arr = np.array([mult.get(lab, 1.0) for lab in T2_LABELS], dtype=np.float32)
        p = p * arr.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p.astype(np.float32)

def adjust_q3(q3, mult):
    p = q3.copy()
    if mult:
        arr = np.array([mult.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * arr.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p.astype(np.float32)

def blend_with_aug_t2(existing_p4, aug_p4, alpha, mult=None):
    aug_p4 = adjust_p4(aug_p4, mult)
    out = (1 - alpha) * existing_p4 + alpha * aug_p4
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def blend_with_aug_t4(existing_q3, aug_q3, alpha, mult=None):
    aug_q3 = adjust_q3(aug_q3, mult)
    out = (1 - alpha) * existing_q3 + alpha * aug_q3
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def timeline_counts(pred):
    return {lab: int((pred["verification_timeline"] == lab).sum()) for lab in LABELS["verification_timeline"]}

def quality_counts(pred):
    return {lab: int((pred["evidence_quality"] == lab).sum()) for lab in LABELS["evidence_quality"]}

def calc_timeline_risk(test_pred, ref_dist):
    cnt = timeline_counts(test_pred)
    already_drop = max(0, ref_dist.get("already", 0) - cnt.get("already", 0))
    more5_inc = max(0, cnt.get("more_than_5_years", 0) - ref_dist.get("more_than_5_years", 0))
    within2_inc = max(0, cnt.get("within_2_years", 0) - ref_dist.get("within_2_years", 0))
    risk = already_drop / 120.0 + more5_inc / 120.0 + within2_inc / 25.0
    return float(risk), {"already_drop": already_drop, "more5_inc": more5_inc, "within2_inc": within2_inc}

def calc_quality_risk(test_pred, ref_dist):
    cnt = quality_counts(test_pred)
    clear_inc = max(0, cnt.get("Clear", 0) - ref_dist.get("Clear", 0))
    not_clear_inc = max(0, cnt.get("Not Clear", 0) - ref_dist.get("Not Clear", 0))
    misleading_inc = max(0, cnt.get("Misleading", 0) - ref_dist.get("Misleading", 0))
    risk = clear_inc / 200.0 + not_clear_inc / 120.0 + misleading_inc / 20.0
    return float(risk), {"clear_inc": clear_inc, "not_clear_inc": not_clear_inc, "misleading_inc": misleading_inc}

def build_v37_pred(df, split, base_choice, t2_aug_cfg=None, t4_aug_cfg=None):
    base_pred, base_probs, existing_p4, existing_q3 = build_existing_pipeline_pred(df, split, base_choice)

    p4 = existing_p4
    q3 = existing_q3

    if t2_aug_cfg is not None and t2_aug_cfg["alpha"] > 0:
        aug_p4 = mix_specialist_views(t2_aug_obj, split, t2_aug_cfg["view"], "t2")
        p4 = blend_with_aug_t2(existing_p4, aug_p4, t2_aug_cfg["alpha"], t2_aug_cfg.get("mult"))

    if t4_aug_cfg is not None and t4_aug_cfg["alpha"] > 0:
        aug_q3 = mix_specialist_views(t4_aug_obj, split, t4_aug_cfg["view"], "t4")
        q3 = blend_with_aug_t4(existing_q3, aug_q3, t4_aug_cfg["alpha"], t4_aug_cfg.get("mult"))

    pred = base_pred.copy()
    pred = compose_timeline_from_p4(pred, p4)
    pred = compose_quality_from_p3(pred, q3)
    pred = apply_hierarchy_to_pred_df(pred)
    return pred


In [ ]:

base_choice_candidates = ["stage3", "stage4_distribution_matched", "stage4_conservative_best_val", "stage4_t2_best_val"]
base_choices = filter_available_stage4_base_choices(base_choice_candidates)
print("Available base choices:", base_choices)

t2_aug_cfgs = [
    {"name": "none", "alpha": 0.0, "view": "head", "mult": None},
    {"name": "t2_a005_head", "alpha": 0.05, "view": "head", "mult": None},
    {"name": "t2_a010_head", "alpha": 0.10, "view": "head", "mult": None},
    {"name": "t2_a020_head", "alpha": 0.20, "view": "head", "mult": None},
    {"name": "t2_a010_head_tail", "alpha": 0.10, "view": "head_tail", "mult": None},
    {"name": "t2_a020_head_tail", "alpha": 0.20, "view": "head_tail", "mult": None},
    {"name": "t2_a030_head_tail", "alpha": 0.30, "view": "head_tail", "mult": None},
    {"name": "t2_a010_within15", "alpha": 0.10, "view": "head_tail", "mult": {"within_2_years": 1.5}},
    {"name": "t2_a020_within15", "alpha": 0.20, "view": "head_tail", "mult": {"within_2_years": 1.5}},
    {"name": "t2_a020_more5_guard", "alpha": 0.20, "view": "head_tail", "mult": {"within_2_years": 1.2, "more_than_5_years": 0.9}},
]

t4_aug_cfgs = [
    {"name": "none", "alpha": 0.0, "view": "head", "mult": None},
    {"name": "t4_a002_head", "alpha": 0.02, "view": "head", "mult": None},
    {"name": "t4_a005_head", "alpha": 0.05, "view": "head", "mult": None},
    {"name": "t4_a010_head", "alpha": 0.10, "view": "head", "mult": None},
    {"name": "t4_a005_notclear", "alpha": 0.05, "view": "head", "mult": {"Not Clear": 1.25, "Misleading": 0.5}},
    {"name": "t4_a010_notclear", "alpha": 0.10, "view": "head", "mult": {"Not Clear": 1.25, "Misleading": 0.5}},
    {"name": "t4_a005_mislead", "alpha": 0.05, "view": "head", "mult": {"Not Clear": 1.1, "Misleading": 1.5}},
    {"name": "t4_a010_mislead", "alpha": 0.10, "view": "head", "mult": {"Not Clear": 1.1, "Misleading": 1.5}},
    {"name": "t4_a005_head_tail", "alpha": 0.05, "view": "head_tail", "mult": {"Not Clear": 1.2, "Misleading": 0.5}},
]

v34_test_pred = build_v37_pred(test_df, "test", "stage3", t2_aug_cfgs[0], t4_aug_cfgs[0])
ref_timeline = timeline_counts(v34_test_pred)
ref_quality = quality_counts(v34_test_pred)

print("ref_timeline:", ref_timeline)
print("ref_quality:", ref_quality)

rows = []
best_val = None
best_safe = None
best_t2_only = None
best_t4_only = None
best_dist = None

for base_choice in base_choices:
    for t2cfg in t2_aug_cfgs:
        for t4cfg in t4_aug_cfgs:
            val_pred = build_v37_pred(val_df, "val", base_choice, t2cfg, t4cfg)
            metrics = calc_metrics(val_df, val_pred)

            test_pred = build_v37_pred(test_df, "test", base_choice, t2cfg, t4cfg)
            trisk, tparts = calc_timeline_risk(test_pred, ref_timeline)
            qrisk, qparts = calc_quality_risk(test_pred, ref_quality)
            total_risk = trisk + qrisk

            row = {
                "base_choice": base_choice,
                "t2_aug": t2cfg["name"],
                "t4_aug": t4cfg["name"],
                **metrics,
                "timeline_risk": trisk,
                "quality_risk": qrisk,
                "total_risk": total_risk,
                **{f"test_timeline_{k}": v for k, v in timeline_counts(test_pred).items()},
                **{f"test_quality_{k}": v for k, v in quality_counts(test_pred).items()},
                **{f"risk_t_{k}": v for k, v in tparts.items()},
                **{f"risk_q_{k}": v for k, v in qparts.items()},
            }
            rows.append(row)

            item = (metrics["weighted_score"], metrics, val_pred, test_pred, base_choice, t2cfg, t4cfg, total_risk, trisk, qrisk)

            if best_val is None or metrics["weighted_score"] > best_val[0]:
                best_val = item

            # 安全：比 stage4_conservative stable 高一點，總風險不要太高
            if metrics["weighted_score"] >= 0.6952 and total_risk <= 1.0:
                if best_safe is None or metrics["weighted_score"] > best_safe[0]:
                    best_safe = item

            # 分布匹配：至少比 v34 高，風險最小
            if metrics["weighted_score"] >= 0.6945:
                if best_dist is None:
                    best_dist = item
                else:
                    if (total_risk < best_dist[7] - 1e-9) or (abs(total_risk - best_dist[7]) < 1e-9 and metrics["weighted_score"] > best_dist[0]):
                        best_dist = item

            if t2cfg["alpha"] > 0 and t4cfg["alpha"] == 0:
                if best_t2_only is None or metrics["weighted_score"] > best_t2_only[0]:
                    best_t2_only = item

            if t2cfg["alpha"] == 0 and t4cfg["alpha"] > 0:
                if best_t4_only is None or metrics["weighted_score"] > best_t4_only[0]:
                    best_t4_only = item

search_df = pd.DataFrame(rows).sort_values(["weighted_score", "total_risk"], ascending=[False, True])
search_df.to_csv(f"{OUT_DIR}/stage5_augmentation_search.csv", index=False, encoding="utf-8-sig")

def show_item(name, item):
    print("\n" + "="*80)
    print(name)
    print("="*80)
    if item is None:
        print("None")
        return
    score, metrics, val_pred, test_pred, base_choice, t2cfg, t4cfg, total_risk, trisk, qrisk = item
    print_metrics(metrics, name)
    print("base_choice:", base_choice)
    print("t2_aug:", t2cfg)
    print("t4_aug:", t4cfg)
    print("total_risk:", total_risk, "timeline_risk:", trisk, "quality_risk:", qrisk)
    print("val timeline:", timeline_counts(val_pred))
    print("test timeline:", timeline_counts(test_pred))
    print("val quality:", quality_counts(val_pred))
    print("test quality:", quality_counts(test_pred))

show_item("Stage5A best_val", best_val)
show_item("Stage5A safe", best_safe)
show_item("Stage5A distribution_matched", best_dist)
show_item("Stage5A best_t2_only", best_t2_only)
show_item("Stage5A best_t4_only", best_t4_only)

display(search_df.head(40))
display(search_df.sort_values(["total_risk", "weighted_score"], ascending=[True, False]).head(40))


## 11. 儲存 val 分析與 config

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

def item_to_config(item):
    if item is None:
        return None
    score, metrics, val_pred, test_pred, base_choice, t2cfg, t4cfg, total_risk, trisk, qrisk = item
    return {
        "base_choice": base_choice,
        "t2_aug": t2cfg,
        "t4_aug": t4cfg,
        "metrics": metrics,
        "total_risk": total_risk,
        "timeline_risk": trisk,
        "quality_risk": qrisk,
        "val_timeline_dist": timeline_counts(val_pred),
        "test_timeline_dist": timeline_counts(test_pred),
        "val_quality_dist": quality_counts(val_pred),
        "test_quality_dist": quality_counts(test_pred),
    }

choices = {
    "best_val": best_val,
    "safe": best_safe,
    "distribution_matched": best_dist,
    "best_t2_only": best_t2_only,
    "best_t4_only": best_t4_only,
}

config_obj = {
    "base": "stage5_model_bank_final_submission",
    "synthetic_counts": syn_pass_df["source_type"].value_counts().to_dict(),
    "t2_aug_fold_scores": t2_aug_obj.get("fold_scores"),
    "t4_aug_fold_scores": t4_aug_obj.get("fold_scores"),
    "reference": {
        "v34_test_timeline": ref_timeline,
        "v34_test_quality": ref_quality,
    },
    "choices": {name: item_to_config(item) for name, item in choices.items()},
}

with open(f"{OUT_DIR}/stage5_augmentation_config.json", "w", encoding="utf-8") as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)

for name, item in choices.items():
    if item is not None:
        _, metrics, val_pred, _, _, _, _, _, _, _ = item
        save_val_outputs(val_pred, metrics, f"stage5_aug_{name}")

print("saved config and val outputs to:", OUT_DIR)
print(json.dumps(config_obj, ensure_ascii=False, indent=2)[:8000])


## 12. 產生 test submissions

In [ ]:

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_submission_from_pred(pred, tag):
    pred = pred.copy()

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)
    sub = pred[["id"] + TASKS].copy()

    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

submission_subs = {}
submission_paths = {}

# base submissions
base_submission_choices = filter_available_stage4_base_choices(["stage3", "stage4_distribution_matched", "stage4_t2_best_val"])
for base_choice in base_submission_choices:
    pred = build_v37_pred(test_df, "test", base_choice, t2_aug_cfgs[0], t4_aug_cfgs[0])
    sub, path = make_submission_from_pred(pred, f"stage5_aug_base_{base_choice}")
    submission_subs[f"base_{base_choice}"] = sub
    submission_paths[f"base_{base_choice}"] = path

for name, item in choices.items():
    if item is None:
        continue
    _, metrics, val_pred, test_pred, base_choice, t2cfg, t4cfg, total_risk, trisk, qrisk = item
    sub, path = make_submission_from_pred(test_pred, f"stage5_aug_{name}")
    submission_subs[name] = sub
    submission_paths[name] = path

print("\nSubmission paths:")
for name, path in submission_paths.items():
    print(name, ":", path)


## 13. 報告摘要

In [ ]:

report = []
report.append("# Stage 5 Minority Augmentation T2/T4 Report")
report.append("")
report.append("## Synthetic data")
report.append(f"- pass count: {len(syn_pass_df)}")
for k, v in syn_pass_df["source_type"].value_counts().to_dict().items():
    report.append(f"- {k}: {v}")

report.append("")
report.append("## Specialist fold scores")
report.append(f"- T2 augmented fold scores: {t2_aug_obj.get('fold_scores')}")
report.append(f"- T4 augmented fold scores: {t4_aug_obj.get('fold_scores')}")

report.append("")
report.append("## Choices")
for name, item in choices.items():
    report.append(f"### {name}")
    cfg = item_to_config(item)
    if cfg is None:
        report.append("- None")
    else:
        report.append(f"- base_choice: {cfg['base_choice']}")
        report.append(f"- t2_aug: {cfg['t2_aug']}")
        report.append(f"- t4_aug: {cfg['t4_aug']}")
        report.append(f"- total_risk: {cfg['total_risk']:.6f}")
        report.append(f"- timeline_risk: {cfg['timeline_risk']:.6f}")
        report.append(f"- quality_risk: {cfg['quality_risk']:.6f}")
        for k, v in cfg["metrics"].items():
            report.append(f"- {k}: {v:.6f}")
        report.append(f"- val_timeline_dist: {cfg['val_timeline_dist']}")
        report.append(f"- test_timeline_dist: {cfg['test_timeline_dist']}")
        report.append(f"- val_quality_dist: {cfg['val_quality_dist']}")
        report.append(f"- test_quality_dist: {cfg['test_quality_dist']}")

report.append("")
report.append("## Submission distributions")
for name, sub in submission_subs.items():
    report.append(f"### {name}")
    for t in TASKS:
        report.append(f"#### {t}")
        for lab, cnt in sub[t].value_counts().to_dict().items():
            report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage5_augmentation_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:10000])
print("saved:", f"{OUT_DIR}/stage5_augmentation_report.md")


## Stage 5B final mini model bank

This section uses public Stage paths and artifact names under `/content/drive/MyDrive/VeriPromiseESG2026`.


## 1. 路徑與設定

In [ ]:

import os
import json
import pickle
import re
import glob
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V32_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_T4_PROB_PATH = f"{V32_DIR}/stage3_t4_specialist_probs.pkl"

V33_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V33_VIEW_PROB_PATH = f"{V33_DIR}/stage3_3view_probs.pkl"

V34_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V34_CONFIG_PATH = f"{V34_DIR}/stage3_best_config.json"

V35_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_T2_PROB_PATH = f"{V35_DIR}/stage4_t2_specialist_probs.pkl"
V35_CONFIG_PATH = f"{V35_DIR}/stage4_t2_best_config.json"

V35_01_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_01_CONFIG_PATH = f"{V35_01_DIR}/stage4_conservative_config.json"

V36_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V36_CONFIG_PATH = f"{V36_DIR}/stage4_config.json"

V37_DIR = f"{BASE_DIR}/outputs/stage5_model_bank_final_submission"
V37_T2_AUG_PROB_PATH = f"{V37_DIR}/stage5_t2_aug_specialist_probs.pkl"
V37_CONFIG_PATH = f"{V37_DIR}/stage5_augmentation_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage5_model_bank_final_submission"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T2_LABELS = ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years"]
T4_LABELS = ["Clear", "Not Clear", "Misleading"]

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

T2_LABEL2ID = {lab: i for i, lab in enumerate(T2_LABELS)}
T2_ID2LABEL = {i: lab for lab, i in T2_LABEL2ID.items()}

T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

print("OUT_DIR:", OUT_DIR)
for p in [
    TRAIN_PATH, VAL_PATH, TEST_PATH,
    V33_VIEW_PROB_PATH, V32_T4_PROB_PATH, V34_CONFIG_PATH,
    V35_T2_PROB_PATH, V35_CONFIG_PATH, V35_01_CONFIG_PATH,
    V36_CONFIG_PATH, V37_T2_AUG_PROB_PATH, V37_CONFIG_PATH
]:
    print(os.path.exists(p), p)


## 2. 讀取資料與快取

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df, test_df]:
    df["id"] = df["id"].astype(str)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(V34_CONFIG_PATH, "r", encoding="utf-8") as f:
    V34_CONFIG_OBJ = json.load(f)
V34_BASE_CONFIG = V34_CONFIG_OBJ["base_config"]
V34_T4_ALPHA = float(V34_CONFIG_OBJ["t4_alpha"])
V34_T4_SPEC_MULT = V34_CONFIG_OBJ["t4_spec_multipliers"]

with open(V33_VIEW_PROB_PATH, "rb") as f:
    view_obj = pickle.load(f)

with open(V32_T4_PROB_PATH, "rb") as f:
    t4_obj = pickle.load(f)
t4_val_probs_old = t4_obj["val_probs"]
t4_test_probs_old = t4_obj["test_probs"]

with open(V35_T2_PROB_PATH, "rb") as f:
    t2_old_obj = pickle.load(f)

with open(V35_CONFIG_PATH, "r", encoding="utf-8") as f:
    V35_CONFIG_OBJ = json.load(f)

with open(V35_01_CONFIG_PATH, "r", encoding="utf-8") as f:
    V35_01_CONFIG_OBJ = json.load(f)

with open(V36_CONFIG_PATH, "r", encoding="utf-8") as f:
    V36_CONFIG_OBJ = json.load(f)

with open(V37_T2_AUG_PROB_PATH, "rb") as f:
    t2_aug_obj = pickle.load(f)

with open(V37_CONFIG_PATH, "r", encoding="utf-8") as f:
    V37_CONFIG_OBJ = json.load(f)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nKnown metrics:")
print("v34:", V34_CONFIG_OBJ.get("best_metrics"))
print("v35:", V35_CONFIG_OBJ.get("best_metrics"))
print("v37:", V37_CONFIG_OBJ.get("choices", {}).get("best_val", {}).get("metrics"))


## 3. Scorer / hierarchy

In [ ]:

def apply_hierarchy_to_pred_df(pred_df):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        pd.Series(y_true).astype(str).values,
        pd.Series(y_pred).astype(str).values,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    pred_df = apply_hierarchy_to_pred_df(pred_df)
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].astype(str).values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(pred)
    return pred


## 4. Reconstruct v34 base + fixed T4

In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

T2_VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

# Compatibility aliases for older Stage3/Stage4 view preset names.
def resolve_view_preset(presets, preset_name, *, default="head", context="view preset"):
    preset_name = default if preset_name is None else preset_name
    aliases = {
        "head_middle": "head_mid",
        "middle_tail": "mid_tail",
    }
    preset_name = aliases.get(preset_name, preset_name)
    if preset_name not in presets:
        available = ", ".join(sorted(presets.keys()))
        raise KeyError(f"Unknown {context}: {preset_name}. Available presets: {available}")
    return presets[preset_name]

def mix_views(stem_view_probs, split, task, preset_name):
    preset = resolve_view_preset(VIEW_PRESETS, preset_name, context="Stage3/Stage4 view preset")
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_config_as_pred(df, split, config):
    probs = build_tta_probs(split, config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(df, probs, post_cfg)
    return pred, probs

def t4_adjust_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_quality_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]

    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]

    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def get_t4_blend_probs(base_probs, base_config, spec_probs, alpha=0.05, spec_multipliers=None):
    base_q3 = get_base_quality_non_na_probs(base_probs, base_config)
    spec_q3 = t4_adjust_probs(spec_probs, spec_multipliers=spec_multipliers)
    blend = (1 - alpha) * base_q3 + alpha * spec_q3
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
    return blend.astype(np.float32)

def compose_quality_from_p3(base_pred, q3):
    out = base_pred.copy()
    mask = out["evidence_status"] == "Yes"
    ids = q3.argmax(axis=1)
    labels = np.array([T4_ID2LABEL[int(i)] for i in ids])
    out.loc[mask, "evidence_quality"] = labels[mask.values]
    out.loc[~mask, "evidence_quality"] = "N/A"
    return apply_hierarchy_to_pred_df(out)

def get_base_timeline_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    p = adjusted["verification_timeline"]

    non_na_ids = [
        LABEL2ID["verification_timeline"]["already"],
        LABEL2ID["verification_timeline"]["within_2_years"],
        LABEL2ID["verification_timeline"]["between_2_and_5_years"],
        LABEL2ID["verification_timeline"]["more_than_5_years"],
    ]

    p4 = p[:, non_na_ids].copy()
    p4 = p4 / np.clip(p4.sum(axis=1, keepdims=True), 1e-12, None)
    return p4.astype(np.float32)

def build_v34_components(df, split):
    base_pred, base_probs = eval_config_as_pred(df, split, V34_BASE_CONFIG)
    t4_probs = t4_val_probs_old if split == "val" else t4_test_probs_old

    q3 = get_t4_blend_probs(
        base_probs,
        V34_BASE_CONFIG,
        t4_probs,
        alpha=V34_T4_ALPHA,
        spec_multipliers=V34_T4_SPEC_MULT,
    )

    pred = compose_quality_from_p3(base_pred, q3)
    base_p4 = get_base_timeline_non_na_probs(base_probs, V34_BASE_CONFIG)
    return pred, base_probs, base_p4, q3

def compose_timeline_from_p4(base_pred, p4):
    out = base_pred.copy()
    mask = out["promise_status"] == "Yes"
    ids = p4.argmax(axis=1)
    labels = np.array([T2_ID2LABEL[int(i)] for i in ids])
    out.loc[mask, "verification_timeline"] = labels[mask.values]
    out.loc[~mask, "verification_timeline"] = "N/A"
    return apply_hierarchy_to_pred_df(out)

def build_final_pred_from_p4(df, split, p4):
    base_pred, base_probs, base_p4, q3 = build_v34_components(df, split)
    pred = compose_timeline_from_p4(base_pred, p4)
    pred = compose_quality_from_p3(pred, q3)
    pred = apply_hierarchy_to_pred_df(pred)
    return pred

# sanity check v34
v34_val_pred, v34_val_base_probs, v34_val_p4, v34_val_q3 = build_v34_components(val_df, "val")
print_metrics(calc_metrics(val_df, v34_val_pred), "reconstructed v34")


## 5. Build T2 mini bank sources

In [ ]:

def mix_t2_views(t2_obj, split, preset_name):
    preset = resolve_view_preset(T2_VIEW_PRESETS, preset_name, context="T2 view preset")
    prob_dict = t2_obj[f"{split}_probs"]

    out = None
    for view, w in preset.items():
        p = prob_dict[view]
        out = w * p if out is None else out + w * p

    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def adjust_p4(p4, mult):
    p = p4.copy()
    if mult:
        arr = np.array([mult.get(lab, 1.0) for lab in T2_LABELS], dtype=np.float32)
        p = p * arr.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p.astype(np.float32)

def blend_p4(items):
    # items: list of (weight, p4)
    out = None
    total = 0.0
    for w, p in items:
        if w <= 0:
            continue
        out = w * p if out is None else out + w * p
        total += w
    if out is None or total <= 0:
        raise ValueError("no positive weights")
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

# cue features copied/reduced from v36
def contains_any(text, patterns):
    return any(re.search(p, text) for p in patterns)

T2_ALREADY_PATTERNS = [
    r"已完成", r"已取得", r"已通過", r"已導入", r"已建置", r"已建立", r"已執行", r"已實施",
    r"完成.{0,8}(查證|驗證|盤查|稽核|認證|確信)",
    r"通過.{0,8}(查證|驗證|認證|稽核)",
    r"取得.{0,8}(認證|證書|證明|查證)",
    r"於\s*\d{4}\s*年.{0,12}(完成|取得|通過|導入|建立|執行)",
]
T2_WITHIN2_PATTERNS = [r"兩年內", r"二年內", r"2\s*年內", r"短期", r"近期", r"明年", r"後年", r"2025\s*年?", r"2026\s*年?"]
T2_BETWEEN25_PATTERNS = [r"未來五年", r"五年內", r"5\s*年內", r"中期", r"中程", r"階段性目標", r"2027\s*年?", r"2028\s*年?", r"2029\s*年?"]
T2_MORE5_PATTERNS = [r"2030\s*年?", r"2035\s*年?", r"2040\s*年?", r"2050\s*年?", r"長期", r"長程", r"淨零", r"碳中和", r"RE100", r"SBTi", r"科學基礎減碳"]

def extract_t2_cues_df(df):
    rows = []
    for text in df["data"].astype(str).tolist():
        compact = re.sub(r"\s+", "", str(text))
        years = [int(y) for y in re.findall(r"(20[2-5]\d)", compact)]
        future_year_max = max(years) if years else None

        cues = {
            "already": contains_any(compact, T2_ALREADY_PATTERNS),
            "within2": contains_any(compact, T2_WITHIN2_PATTERNS),
            "between25": contains_any(compact, T2_BETWEEN25_PATTERNS),
            "more5": contains_any(compact, T2_MORE5_PATTERNS),
        }

        if future_year_max is not None:
            if future_year_max >= 2030:
                cues["more5"] = True
            elif 2027 <= future_year_max <= 2029:
                cues["between25"] = True
            elif 2025 <= future_year_max <= 2026:
                cues["within2"] = True

        rows.append(cues)
    return pd.DataFrame(rows)

T2_CUE_PRESETS = {
    "none": {"already": 1.0, "within2": 1.0, "between25": 1.0, "more5": 1.0},
    "within2_focus": {"already": 1.0, "within2": 1.35, "between25": 1.0, "more5": 1.0},
    "balanced_t2": {"already": 1.08, "within2": 1.15, "between25": 1.10, "more5": 1.10},
    "timeline_years": {"already": 1.0, "within2": 1.20, "between25": 1.15, "more5": 1.18},
    "already_guard": {"already": 1.12, "within2": 1.10, "between25": 1.06, "more5": 1.06},
}

def apply_t2_cues(p4, cues, preset_name):
    preset = T2_CUE_PRESETS[preset_name]
    out = p4.copy()
    mult = np.ones_like(out, dtype=np.float32)

    mult[:, T2_LABEL2ID["already"]] *= np.where(cues["already"].values, preset["already"], 1.0)
    mult[:, T2_LABEL2ID["within_2_years"]] *= np.where(cues["within2"].values, preset["within2"], 1.0)
    mult[:, T2_LABEL2ID["between_2_and_5_years"]] *= np.where(cues["between25"].values, preset["between25"], 1.0)
    mult[:, T2_LABEL2ID["more_than_5_years"]] *= np.where(cues["more5"].values, preset["more5"], 1.0)

    out = out * mult
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

val_cues = extract_t2_cues_df(val_df)
test_cues = extract_t2_cues_df(test_df)

print("val cue rates:")
display(val_cues.mean().to_frame("val").T)
print("test cue rates:")
display(test_cues.mean().to_frame("test").T)


In [ ]:
def get_existing_t2_cfg(choice_name):
    original_choice = choice_name
    choice_name = normalize_stage4_base_choice(choice_name)

    if choice_name == "stage4_t2_best_val":
        return {
            "view": V35_CONFIG_OBJ["best_view"],
            "alpha": V35_CONFIG_OBJ["best_alpha"],
            "spec_multipliers": V35_CONFIG_OBJ["best_spec_multipliers"],
        }

    if choice_name.startswith("stage4_conservative_"):
        name = choice_name.replace("stage4_conservative_", "")
        cfg = V35_01_CONFIG_OBJ.get("choices", {}).get(name)
        if cfg is None:
            raise ValueError(f"choice not found or None: {original_choice} -> {choice_name}")
        return {
            "view": cfg["view"],
            "alpha": cfg["alpha"],
            "spec_multipliers": cfg["spec_multipliers"],
        }

    if choice_name == "v34":
        return {"view": "head", "alpha": 0.0, "spec_multipliers": None}

    raise ValueError(f"unknown choice_name: {original_choice} -> {choice_name}")

def build_existing_t2_blend(split, base_p4, choice_name):
    cfg = get_existing_t2_cfg(choice_name)
    if cfg["alpha"] <= 0:
        return base_p4.copy()

    spec = mix_t2_views(t2_old_obj, split, cfg["view"])
    spec = adjust_p4(spec, cfg["spec_multipliers"])
    out = (1 - cfg["alpha"]) * base_p4 + cfg["alpha"] * spec
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def build_stage5_aug_best_t2(split, base_p4):
    choice = V37_CONFIG_OBJ["choices"]["best_val"]
    cfg = choice["t2_aug"]
    base_choice = choice.get("base_choice", "stage4_t2_best_val")

    p4_base = build_existing_t2_blend(split, base_p4, base_choice)
    aug = mix_t2_views(t2_aug_obj, split, cfg["view"])
    aug = adjust_p4(aug, cfg.get("mult"))
    out = (1 - cfg["alpha"]) * p4_base + cfg["alpha"] * aug
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def build_stage5_aug_safe_t2(split, base_p4):
    choice = V37_CONFIG_OBJ.get("choices", {}).get("safe") or V37_CONFIG_OBJ.get("choices", {}).get("best_val")
    if choice is None:
        return build_stage5_aug_best_t2(split, base_p4)

    cfg = choice["t2_aug"]
    base_choice = choice.get("base_choice", "stage4_t2_best_val")

    p4_base = build_existing_t2_blend(split, base_p4, base_choice)
    aug = mix_t2_views(t2_aug_obj, split, cfg["view"])
    aug = adjust_p4(aug, cfg.get("mult"))
    out = (1 - cfg["alpha"]) * p4_base + cfg["alpha"] * aug
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def build_t2_source_bank(split):
    if split == "val":
        df = val_df
        base_pred, base_probs, base_p4, q3 = build_v34_components(df, split)
        cues = val_cues
    else:
        df = test_df
        base_pred, base_probs, base_p4, q3 = build_v34_components(df, split)
        cues = test_cues

    sources = {}

    sources["base_v34"] = base_p4
    sources["stage4_t2_best_val"] = build_existing_t2_blend(split, base_p4, "stage4_t2_best_val")

    optional_stage4_sources = {
        "stage4_distribution_matched": "stage4_distribution_matched",
        "stage4_conservative_best_val": "stage4_conservative_best_val",
    }

    for source_name, choice_name in optional_stage4_sources.items():
        if is_stage4_base_choice_available(choice_name):
            sources[source_name] = build_existing_t2_blend(split, base_p4, choice_name)
        else:
            print(f"Skipping unavailable source on {split}: {source_name}")

    sources["old_spec_head"] = mix_t2_views(t2_old_obj, split, "head")
    sources["old_spec_head_tail"] = mix_t2_views(t2_old_obj, split, "head_tail")
    sources["old_spec_tail"] = mix_t2_views(t2_old_obj, split, "tail")

    sources["aug_spec_head"] = mix_t2_views(t2_aug_obj, split, "head")
    sources["aug_spec_head_tail"] = mix_t2_views(t2_aug_obj, split, "head_tail")
    sources["aug_spec_tail"] = mix_t2_views(t2_aug_obj, split, "tail")

    sources["stage5_aug_best"] = build_stage5_aug_best_t2(split, base_p4)
    sources["stage5_aug_safe"] = build_stage5_aug_safe_t2(split, base_p4)

    sources["stage4_t2_best_val_cue_within2"] = apply_t2_cues(sources["stage4_t2_best_val"], cues, "within2_focus")
    sources["stage4_t2_best_val_cue_balanced"] = apply_t2_cues(sources["stage4_t2_best_val"], cues, "balanced_t2")
    sources["stage5_aug_best_cue_within2"] = apply_t2_cues(sources["stage5_aug_best"], cues, "within2_focus")
    sources["stage5_aug_best_cue_guard"] = apply_t2_cues(sources["stage5_aug_best"], cues, "already_guard")

    return sources, base_p4

val_sources, val_base_p4 = build_t2_source_bank("val")
test_sources, test_base_p4 = build_t2_source_bank("test")

print("T2 sources:", list(val_sources.keys()))

# source diagnostics
source_rows = []
for name, p4 in val_sources.items():
    pred = build_final_pred_from_p4(val_df, "val", p4)
    m = calc_metrics(val_df, pred)
    row = {
        "source": name,
        **m,
        "timeline_dist": pred["verification_timeline"].value_counts().to_dict(),
    }
    source_rows.append(row)

source_df = pd.DataFrame(source_rows).sort_values("weighted_score", ascending=False)
source_df.to_csv(f"{OUT_DIR}/stage5_t2_source_diagnostics.csv", index=False, encoding="utf-8-sig")
display(source_df)

## 6. Mini model bank search



In [ ]:

def timeline_counts(pred):
    return {lab: int((pred["verification_timeline"] == lab).sum()) for lab in LABELS["verification_timeline"]}

def quality_counts(pred):
    return {lab: int((pred["evidence_quality"] == lab).sum()) for lab in LABELS["evidence_quality"]}

def calc_timeline_risk(test_pred, ref_dist):
    cnt = timeline_counts(test_pred)
    already_drop = max(0, ref_dist.get("already", 0) - cnt.get("already", 0))
    more5_inc = max(0, cnt.get("more_than_5_years", 0) - ref_dist.get("more_than_5_years", 0))
    within2_inc = max(0, cnt.get("within_2_years", 0) - ref_dist.get("within_2_years", 0))
    risk = already_drop / 120.0 + more5_inc / 120.0 + within2_inc / 25.0
    return float(risk), {
        "already_drop": int(already_drop),
        "more5_inc": int(more5_inc),
        "within2_inc": int(within2_inc),
    }

# reference v34
v34_test_pred = build_final_pred_from_p4(test_df, "test", test_sources["base_v34"])
ref_timeline = timeline_counts(v34_test_pred)
ref_quality = quality_counts(v34_test_pred)

print("ref_timeline:", ref_timeline)
print("ref_quality:", ref_quality)

def make_candidate_weights():
    candidates = []

    # single sources
    for s in [
        "base_v34",
        "stage4_t2_best_val",
        "stage4_distribution_matched",
        "stage4_conservative_best_val",
        "stage5_aug_best",
        "stage5_aug_safe",
        "stage4_t2_best_val_cue_within2",
        "stage5_aug_best_cue_within2",
    ]:
        candidates.append({"name": s, "weights": {s: 1.0}})

    # Two-way blends around known strong variants
    blend_pairs = [
        ("stage5_aug_best", "stage4_t2_best_val"),
        ("stage5_aug_best", "v36_like_cue"),
        ("stage5_aug_best", "stage4_distribution_matched"),
        ("stage5_aug_best", "stage5_aug_safe"),
        ("stage5_aug_best", "base_v34"),
        ("stage4_t2_best_val_cue_within2", "stage5_aug_best"),
        ("stage5_aug_best_cue_within2", "stage5_aug_best"),
        ("old_spec_head", "aug_spec_head"),
        ("stage4_t2_best_val", "aug_spec_head"),
        ("stage4_distribution_matched", "aug_spec_head"),
    ]

    # Alias v36_like_cue to stage4_t2_best_val_cue_within2
    alias = {"v36_like_cue": "stage4_t2_best_val_cue_within2"}

    for a, b in blend_pairs:
        a2 = alias.get(a, a)
        b2 = alias.get(b, b)
        for wa in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
            wb = 1 - wa
            name = f"{a2}_{wa:.1f}__{b2}_{wb:.1f}"
            candidates.append({"name": name, "weights": {a2: wa, b2: wb}})

    # Three-way blends: base / stable / aggressive
    triplets = [
        ("base_v34", "stage4_distribution_matched", "stage5_aug_best"),
        ("stage4_distribution_matched", "stage4_t2_best_val", "stage5_aug_best"),
        ("stage4_t2_best_val", "stage5_aug_best", "aug_spec_head"),
        ("stage4_t2_best_val", "stage5_aug_best", "stage5_aug_best_cue_within2"),
        ("base_v34", "stage5_aug_safe", "stage5_aug_best"),
    ]

    grid = [
        (0.2, 0.3, 0.5),
        (0.2, 0.4, 0.4),
        (0.3, 0.3, 0.4),
        (0.1, 0.4, 0.5),
        (0.1, 0.3, 0.6),
        (0.4, 0.3, 0.3),
        (0.5, 0.2, 0.3),
    ]

    for a, b, c in triplets:
        for wa, wb, wc in grid:
            name = f"{a}_{wa:.1f}__{b}_{wb:.1f}__{c}_{wc:.1f}"
            candidates.append({"name": name, "weights": {a: wa, b: wb, c: wc}})

    # Deduplicate by name
    seen = set()
    out = []
    for c in candidates:
        if c["name"] not in seen:
            out.append(c)
            seen.add(c["name"])
    return out

def filter_candidates_by_available_sources(candidates, *source_dicts):
    available = []
    skipped = []

    for cand in candidates:
        missing = sorted({
            name
            for name in cand["weights"]
            if any(name not in source_dict for source_dict in source_dicts)
        })

        if missing:
            skipped.append((cand["name"], missing))
        else:
            available.append(cand)

    if skipped:
        print(f"Skipping {len(skipped)} candidates with unavailable sources.")
        print("First skipped candidates:", skipped[:10])

    if not available:
        raise RuntimeError("No model-bank candidates remain after source filtering.")

    return available

candidates = filter_candidates_by_available_sources(
    make_candidate_weights(),
    val_sources,
    test_sources,
)
print("candidate count:", len(candidates))

def build_p4_from_weights(source_dict, weights):
    items = []
    for name, w in weights.items():
        if name not in source_dict:
            raise KeyError(name)
        items.append((w, source_dict[name]))
    return blend_p4(items)

rows = []
best_val = None
best_safe = None
best_dist = None
best_low_risk_high = None

for cand in candidates:
    name = cand["name"]
    weights = cand["weights"]

    val_p4 = build_p4_from_weights(val_sources, weights)
    val_pred = build_final_pred_from_p4(val_df, "val", val_p4)
    metrics = calc_metrics(val_df, val_pred)

    test_p4 = build_p4_from_weights(test_sources, weights)
    test_pred = build_final_pred_from_p4(test_df, "test", test_p4)
    trisk, tparts = calc_timeline_risk(test_pred, ref_timeline)

    row = {
        "candidate": name,
        "weights_json": json.dumps(weights, ensure_ascii=False),
        **metrics,
        "timeline_risk": trisk,
        "total_risk": trisk,
        **{f"test_timeline_{k}": v for k, v in timeline_counts(test_pred).items()},
        **{f"test_quality_{k}": v for k, v in quality_counts(test_pred).items()},
        **{f"risk_{k}": v for k, v in tparts.items()},
    }
    rows.append(row)

    item = (metrics["weighted_score"], metrics, val_pred, test_pred, cand, trisk)

    if best_val is None or metrics["weighted_score"] > best_val[0]:
        best_val = item

    # safe：至少接近/高於 stage5_aug_safe，risk 不超過 1.0
    if metrics["weighted_score"] >= 0.6960 and trisk <= 1.0:
        if best_safe is None or metrics["weighted_score"] > best_safe[0]:
            best_safe = item

    # distribution matched：至少比 v34 高，risk 越低越好
    if metrics["weighted_score"] >= 0.6940:
        if best_dist is None:
            best_dist = item
        else:
            if trisk < best_dist[5] - 1e-9 or (abs(trisk - best_dist[5]) < 1e-9 and metrics["weighted_score"] > best_dist[0]):
                best_dist = item

    # low-risk high score：分數 >= 0.698 且 risk <= 1.5
    if metrics["weighted_score"] >= 0.6980 and trisk <= 1.5:
        if best_low_risk_high is None or metrics["weighted_score"] > best_low_risk_high[0]:
            best_low_risk_high = item

search_df = pd.DataFrame(rows).sort_values(["weighted_score", "total_risk"], ascending=[False, True])
search_df.to_csv(f"{OUT_DIR}/stage5_model_bank_search.csv", index=False, encoding="utf-8-sig")

def show_item(title, item):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    if item is None:
        print("None")
        return
    score, metrics, val_pred, test_pred, cand, risk = item
    print_metrics(metrics, title)
    print("candidate:", cand["name"])
    print("weights:", cand["weights"])
    print("timeline_risk:", risk)
    print("val timeline:", timeline_counts(val_pred))
    print("test timeline:", timeline_counts(test_pred))
    print("test quality:", quality_counts(test_pred))

show_item("v39 best_val", best_val)
show_item("v39 safe", best_safe)
show_item("v39 distribution_matched", best_dist)
show_item("v39 low_risk_high_score", best_low_risk_high)

display(search_df.head(50))
display(search_df.sort_values(["total_risk", "weighted_score"], ascending=[True, False]).head(50))


## 7. Save val outputs / config

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"] + ([ "pdf_url"] if "pdf_url" in val_df.columns else [])].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

def item_to_config(item):
    if item is None:
        return None
    score, metrics, val_pred, test_pred, cand, risk = item
    return {
        "candidate": cand["name"],
        "weights": cand["weights"],
        "metrics": metrics,
        "timeline_risk": risk,
        "val_timeline_dist": timeline_counts(val_pred),
        "test_timeline_dist": timeline_counts(test_pred),
        "test_quality_dist": quality_counts(test_pred),
    }

choices = {
    "best_val": best_val,
    "safe": best_safe,
    "distribution_matched": best_dist,
    "low_risk_high_score": best_low_risk_high,
}

config_obj = {
    "base": "stage5_model_bank_final_submission",
    "note": "T2-focused mini model bank; T4 fixed from v34/v32 blend.",
    "source_names": list(val_sources.keys()),
    "choices": {name: item_to_config(item) for name, item in choices.items()},
    "reference": {
        "v34_test_timeline": ref_timeline,
        "v34_test_quality": ref_quality,
    }
}

with open(f"{OUT_DIR}/stage5_config.json", "w", encoding="utf-8") as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)

for name, item in choices.items():
    if item is not None:
        _, metrics, val_pred, _, _, _ = item
        save_val_outputs(val_pred, metrics, f"stage5_{name}")

print("saved config and val outputs to:", OUT_DIR)
print(json.dumps(config_obj, ensure_ascii=False, indent=2)[:9000])


## 8. Generate test submissions

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_submission_from_pred(pred, tag):
    pred = pred.copy()

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)
    sub = pred[["id"] + TASKS].copy()

    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

submission_subs = {}
submission_paths = {}

# Important existing-like baselines generated in this notebook
submission_source_names = [
    name
    for name in ["base_v34", "stage4_distribution_matched", "stage4_t2_best_val", "stage5_aug_best", "stage5_aug_safe"]
    if name in test_sources]

print("Submission source names:", submission_source_names)

for source_name in submission_source_names:
    pred = build_final_pred_from_p4(test_df, "test", test_sources[source_name])
    sub, path = make_submission_from_pred(pred, f"stage5_base_{source_name}")
    submission_subs[f"base_{source_name}"] = sub
    submission_paths[f"base_{source_name}"] = path

for name, item in choices.items():
    if item is None:
        continue
    _, metrics, val_pred, test_pred, cand, risk = item
    sub, path = make_submission_from_pred(test_pred, f"stage5_{name}")
    submission_subs[name] = sub
    submission_paths[name] = path

print("\nSubmission paths:")
for name, path in submission_paths.items():
    print(name, ":", path)


## 9. Report

In [ ]:

report = []
report.append("# Stage 5 Model Bank 最終提交報告")
report.append("")
report.append("## T2 source diagnostics")
for _, row in source_df.head(20).iterrows():
    report.append(
        f"- `{row['source']}`: weighted={row['weighted_score']:.6f}, "
        f"T2={row['verification_timeline_macro_f1']:.6f}, "
        f"T4={row['evidence_quality_macro_f1']:.6f}, "
        f"timeline={row['timeline_dist']}"
    )

report.append("")
report.append("## Choices")
for name, item in choices.items():
    report.append(f"### {name}")
    cfg = item_to_config(item)
    if cfg is None:
        report.append("- None")
    else:
        report.append(f"- candidate: `{cfg['candidate']}`")
        report.append(f"- weights: `{cfg['weights']}`")
        report.append(f"- timeline_risk: {cfg['timeline_risk']:.6f}")
        for k, v in cfg["metrics"].items():
            report.append(f"- {k}: {v:.6f}")
        report.append(f"- val_timeline_dist: {cfg['val_timeline_dist']}")
        report.append(f"- test_timeline_dist: {cfg['test_timeline_dist']}")
        report.append(f"- test_quality_dist: {cfg['test_quality_dist']}")

report.append("")
report.append("## Submission distributions")
for name, sub in submission_subs.items():
    report.append(f"### {name}")
    for t in TASKS:
        report.append(f"#### {t}")
        for lab, cnt in sub[t].value_counts().to_dict().items():
            report.append(f"- {lab}: {cnt}")

report.append("")
report.append("## Files")
report.append(f"- search: `{OUT_DIR}/stage5_model_bank_search.csv`")
report.append(f"- source diagnostics: `{OUT_DIR}/stage5_t2_source_diagnostics.csv`")
report.append(f"- config: `{OUT_DIR}/stage5_config.json`")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage5_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:12000])
print("saved:", f"{OUT_DIR}/stage5_report.md")
